# Laboratorio de regresión logística

|                |   |
:----------------|---|
| **Nombre**     | Carlos Enrique Nieves Ochoa |
| **Fecha**      | 11 de octubre de 2026 |
| **Expediente** | 758210 |

In machine learning, Support Vector Machines (SVM) are supervised learning models with associated learning algorithms that analyze data used for classification and regression analysis. It is mostly used in classification problems. In this algorithm, each data item is plotted as a point in p-dimensional space (where p is the number of features), with the value of each feature being the value of a particular coordinate. Then, classification is performed by finding the hyper-plane that best differentiates the two classes (or more if we have a multi class problem):

$$ f(x) = w^T \varphi(x) + b $$

where $\varphi: X \rightarrow F $ is a function that makes each input point $x$ correspond to a point in F, where F is a Hilbert space.

In addition to performing linear classification, SVMs can efficiently perform a non-linear classification, implicitly mapping their inputs into high-dimensional feature spaces (more specifically using the kernel trick, like the RBF funcion). 

[1]

OLS utilizes the squared residuals to fit the parameters. Large residuals caused by outliers may worsen the accuracy significantly.

Support Vectors use piecewise linear functions to counter this, in which a hyperparameter  $\epsilon$ called the margin lets errors that are less or equal to it be 0, and error larger than it be $e - \epsilon$. 

The problem to solve is:

\begin{split}
        \min_{w, b, \xi, \xi^*} \mathcal{P}_\epsilon(w, b, \xi) &= \frac{1}{2} w^T w + c \sum_{k=1}^{N} \xi_k \\
        \text{s.t. } & y_k [w^T \varphi(x_k) - b] \geq 1- \xi_k,\ \ k = 1, ..., N \\
        & \xi_k \geq 0,\ \ k = 1, ..., N
\end{split}


The most important question that arises when using a SVM is how to choose the correct hyperplane. Consider the following scenarios:

### Scenario 1

In this scenario there are three hyperplanes called A, B, and C. Now, the problem is to identify the hyperplane which best differentiates the stars and the circles.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_21-2.png" alt="what image shows"></center>

In this case, hyperplane B separates the stars and the circle betters, hence it is the correct hyperplane.


### Scenario 2

Now take another scenario where all three hyperplanes are segregating classes well. The question that arises is how to choose the best hyperplane in this situation.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_4-2.png" alt="what image shows"></center>

In such scenarios, we calculate the margin (which is the distance between nearest data point and the hyperplane). The hyperplane with the largest margin will be considered as the correct hyperplane to classify the dataset.

Here C has the largest margin. Hence, it is considered as the best hyperplane.


### Kernels
Knowing 
$$ w = \sum_{k=1}^{N} \alpha_k y_k \varphi(x_k) $$

And
$$ y_{pred} = w^T \varphi(x) + b $$

Then 
$$ y_{pred} = (\sum_{k=1}^{N} \alpha_k y_k \varphi(x_k))^T \varphi(x) + b $$

Where $\varphi$ is a function that makes each input in $x$ correspond to a point in $F$ (a Hilbert space). This can be seen as processing and transforming the input featuers to keep the model's convexity. [2]

This also allows us to transform the inputs into another space where they might be more easily classified.

<center><img src=https://miro.medium.com/max/838/1*gXvhD4IomaC9Jb37tzDUVg.png alt="what image shows"></center>

## ROC and AUC

A ROC (Receiver Operating Characteristic) is a graph that shows how the classification model performs at the classification thresholds. 

ROC curves typically feature true positive rate on the Y axis, and false positive rate on the X axis. This means that the top left corner of the plot is the “ideal” point - a false positive rate of zero, and a true positive rate of one. This is not very realistic, but it does mean that a larger area under the curve (AUC) is usually better. [3]

True Positive Rate is a synonym for Recall and defined as:
$$ TPR = \frac{TP}{TP + FN} $$

False Positive Rate is a synonym for Specificity and defined as:

$$ FPR = \frac{FP}{FP + TN} $$

ROC curves are typically used in binary classification to study the output of a classifier. In order to extend ROC curve and ROC area to multi-label classification, it is necessary to binarize the output. One ROC curve can be drawn per label, but one can also draw a ROC curve by considering each element of the label indicator matrix as a binary prediction (micro-averaging).

E.g. If you lower a classification threshold, more items would be classified as positive, increasing False Positives and True Positives.

AUC stands for Area under the ROC.

## Ejercicio 1

- Utiliza el dataset `Iris`, modela con SVC y haz Cross-Validation de diferentes kernels ('linear', 'poly', 'rbf', 'sigmoid').
- Modela con LogisticRegression.
- El método de Cross-Validation es K-Folds con $k=10$.
- Utiliza el AUC como métrico de Cross-Validation.
- Compara resultados.

In [ ]:
# Librerías
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

In [2]:
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Iris viene ordenado por clase, por eso mezclo antes de dividir
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# probability=True permite usar predict_proba para calcular AUC
modelos_iris = [
    ("SVC linear", SVC(kernel="linear", probability=True, random_state=42)),
    ("SVC poly", SVC(kernel="poly", probability=True, random_state=42)),
    ("SVC rbf", SVC(kernel="rbf", probability=True, random_state=42)),
    ("SVC sigmoid", SVC(kernel="sigmoid", probability=True, random_state=42)),
    ("LogisticRegression", LogisticRegression(max_iter=1000))
]

In [3]:
resultados_iris = []

for nombre, modelo in modelos_iris:
    auc = []

    for train, test in kf.split(X_iris):
        # El escalador se ajusta solo con el grupo de entrenamiento
        pipeline = Pipeline(steps=[
            ("scaler", StandardScaler()),
            ("model", modelo)
        ])

        pipeline.fit(X_iris[train], y_iris[train])
        prob = pipeline.predict_proba(X_iris[test])

        # AUC de cada clase contra las demás, luego promedio las tres
        auc.append(roc_auc_score(
            y_iris[test], prob, multi_class="ovr", average="macro"
        ))

    resultados_iris.append([nombre, np.mean(auc), np.std(auc)])

tabla_iris = pd.DataFrame(
    resultados_iris,
    columns=["Modelo", "AUC promedio", "Desviación estándar"]
)
tabla_iris.round(4)

c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\

,Modelo,AUC promedio,Desviación estándar
0,SVC linear,1.0000,0.0000
1,SVC poly,1.0000,0.0000
2,SVC rbf,0.9959,0.0124
3,SVC sigmoid,0.9819,0.0333
4,LogisticRegression,0.9967,0.0082


**Comparación de resultados**

En Iris, SVC con kernel linear y poly empatan con AUC promedio de 1 y desviación de 0, les siguen la regresión logística con 0.9967 y rbf con 0.9959, mientras que sigmoid obtiene 0.9819 y presenta la mayor variación entre grupos, para estas particiones me quedo con el kernel lineal por ser el más sencillo de los dos que empatan, un AUC de 1 no significa que todas las etiquetas predichas sean correctas.

## Ejercicio 2
- Repite el ejercicio 1 con el dataset `Default`. Utiliza `default` como target.

In [4]:
df = pd.read_csv("Default.csv", index_col=0)

X_default = df[["balance", "income", "student"]].copy()
X_default["student"] = (df["student"] == "Yes").astype(int)
y_default = (df["default"] == "Yes").astype(int)

numerical_cols = ["balance", "income"]
boolean_cols = ["student"]

modelos_default = [
    ("SVC linear", SVC(kernel="linear", probability=True, random_state=42)),
    ("SVC poly", SVC(kernel="poly", probability=True, random_state=42)),
    ("SVC rbf", SVC(kernel="rbf", probability=True, random_state=42)),
    ("SVC sigmoid", SVC(kernel="sigmoid", probability=True, random_state=42)),
    ("LogisticRegression", LogisticRegression(max_iter=1000))
]

In [5]:
resultados_default = []

for nombre, modelo in modelos_default:
    auc = []

    for train, test in kf.split(X_default):
        preprocessor = ColumnTransformer(transformers=[
            ("num", StandardScaler(), numerical_cols),
            ("bool", "passthrough", boolean_cols)
        ])
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", modelo)
        ])

        pipeline.fit(X_default.iloc[train], y_default.iloc[train])

        # La segunda columna es la probabilidad de default = Yes
        prob = pipeline.predict_proba(X_default.iloc[test])[:, 1]
        auc.append(roc_auc_score(y_default.iloc[test], prob))

    resultados_default.append([nombre, np.mean(auc), np.std(auc)])

tabla_default = pd.DataFrame(
    resultados_default,
    columns=["Modelo", "AUC promedio", "Desviación estándar"]
)
tabla_default.round(4)

c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\

,Modelo,AUC promedio,Desviación estándar
0,SVC linear,0.9300,0.0313
1,SVC poly,0.8906,0.0292
2,SVC rbf,0.7784,0.0460
3,SVC sigmoid,0.8012,0.0188
4,LogisticRegression,0.9491,0.0171


**Comparación de resultados**

En Default, la regresión logística obtiene el mayor AUC promedio con 0.9491 y la menor desviación con 0.0171, por eso la elijo entre los modelos probados, SVC linear queda después con 0.93, seguido de poly con 0.8906, sigmoid con 0.8012 y rbf con 0.7784, estos resultados corresponden a los parámetros utilizados y a los mismos diez grupos de validación.

# Addendum

Métricos disponibles para clasificación:
- ‘accuracy’
- ‘balanced_accuracy’
- ‘top_k_accuracy’
- ‘average_precision’
- ‘neg_brier_score’
- ‘f1’
- ‘f1_micro’
- ‘f1_macro’
- ‘f1_weighted’
- ‘f1_samples’
- ‘neg_log_loss’
- ‘precision’ etc.
- ‘recall’ etc.
- ‘jaccard’ etc.
- ‘roc_auc’
- ‘roc_auc_ovr’
- ‘roc_auc_ovo’
- ‘roc_auc_ovr_weighted’
- ‘roc_auc_ovo_weighted’
- ‘d2_log_loss_score’

# References

[1] Shigeo Abe.Support Vector Machines for Pattern Classification,2Ed.Springer-Verlag London,2010. ISBN978-1-84996-097-7. URLhttps://www.springer.com/gp/book/9781849960977.

[2] Johan A K Suykens, Tony Van Gestel, Jos De Brabanter, BartDe Moor, and Joos Vandewalle.Least Squares Support VectorMachines. World Scientific,2002. ISBN9789812381514. URLhttps://www.worldscientific.com/worldscibooks/10.1142/5089.

[3] Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. Pattern recognition, 30(7), 1145-1159. URL https://www.researchgate.net/post/how_can_I_interpret_the_ROC_curve_result

In [3]:
from sklearn.preprocessing import OneHotEncoder

X_rbf = df[["balance", "income", "student"]].copy()
y_rbf = (df["default"] == "Yes").astype(int)

kf_rbf = KFold(n_splits=10, shuffle=False)
auc_rbf = []

for train_rbf, test_rbf in kf_rbf.split(X_rbf):
    preprocessor_rbf = ColumnTransformer(transformers=[
        ("num", StandardScaler(), ["balance", "income"]),
        ("cat", OneHotEncoder(), ["student"])
    ])

    pipeline_rbf = Pipeline(steps=[
        ("preprocessor", preprocessor_rbf),
        ("model", SVC(kernel="rbf", probability=True, random_state=42))
    ])

    pipeline_rbf.fit(X_rbf.iloc[train_rbf], y_rbf.iloc[train_rbf])
    prob_rbf = pipeline_rbf.predict_proba(X_rbf.iloc[test_rbf])[:, 1]
    auc_rbf.append(roc_auc_score(y_rbf.iloc[test_rbf], prob_rbf))

# Error estándar de la media, usando la desviación muestral
error_estandar_rbf = np.std(auc_rbf, ddof=1) / np.sqrt(len(auc_rbf))
print("Error estándar de la media del AUC:", error_estandar_rbf)

C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Error estándar de la media del AUC: 0.009223432278732055
